<a href="https://colab.research.google.com/github/mgd-code/llm-workshop-witten/blob/main/climate_classification_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automating classification with Anthropic models: Central bank climate speeches


**Before running:**
1. Save a local copy of this .ipynb file in your Google drive so you can access it again later
2. Click the key icon in the left sidebar (**Secrets**), add a secret named `ANTHROPIC_API_KEY` with your key as the value, and switch on **Notebook access**.
3. No GPU is needed for this notebook, so the default CPU runtime is fine (**Runtime → Change runtime type**).

## 1. Basic setup: install and load packages


In [ ]:
# Install the Anthropic library
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 11.6 MB/s  0:00:00


In [ ]:
# Import packages that are already installed

import pandas as pd # pandas package used for data analysis
import anthropic # anthropic package
from pathlib import Path # helps with calling different folder and file names
from google.colab import userdata, drive, files # allows us to call the API key and other Google Drive files

Mounted at /content/drive


In [ ]:
# Import the API key from Colab Secrets
api_key = userdata.get("ANTHROPIC_API_KEY")

## 2. Load the data from Google Drive


In [ ]:
# Download the workshop files into this Colab session and set the project_root
!git clone https://github.com/YOUR_USERNAME/llm-workshop.git
project_root = Path("/content/llm-workshop")

In [ ]:
# load climate datasets
climate_dataset_10 = pd.read_csv(project_root/"climate_sample_10.csv")
climate_dataset_100 = pd.read_csv(project_root/"climate_sample_100.csv")

In [ ]:
# check to make sure files loaded

climate_dataset_10

,excerpt_id,contains_keyword,keywords_found,text,date,speaker,central_bank,title,description,url
0,1,0,NaN,Volatility is currently high in global capital...,11/4/2015,Benoît Cœuré,European Central Bank,Benoît Cœuré: Interview in El Financiero,"Interview with Mr Benoît Cœuré, Member of the ...",https://www.bis.org/review/r151117e.htm
1,2,1,climate change,There are a wealth of initiatives at present: ...,7/1/2019,Margarita Delgado,Bank of Spain,Margarita Delgado: Banks and society. Looking ...,"Speech by Ms Margarita Delgado, Deputy Governo...",https://www.bis.org/review/r190716h.htm
2,3,1,climate change,Only through robust and inclusive internationa...,10/1/2025,Jose Luis Escrivá,Bank of Spain,José Luis Escrivá: Introductory remarks - 9th ...,"Introductory remarks by Mr José Luis Escrivá, ...",https://www.bis.org/review/r251002e.htm
3,4,1,climate change,"But much remains to be done. In this respect, ...",7/16/2019,François Villeroy de Galhau,Bank of France,François Villeroy de Galhau: 75 years later - ...,Welcome address by Mr François Villeroy de Gal...,https://www.bis.org/review/r190716k.htm
4,5,0,NaN,The implementation of the eurosystem's asset p...,5/9/2018,Philip R Lane,Central Bank of Ireland,Philip R Lane: Publication of the Annual Repor...,"Introductory statement by Mr Philip R Lane, Go...",https://www.bis.org/review/r180727c.htm
5,6,1,climate change,"Rising protectionism, shifting geopolitical al...",9/16/2025,Yannis Stournaras,Bank of Greece,Yannis Stournaras: Greece as a strategic inves...,"Speech by Mr Yannis Stournaras, Governor of th...",https://www.bis.org/review/r250918d.htm
6,7,1,climate change,"Let me now come to the ECB's strategic review,...",9/25/2020,François Villeroy de Galhau,Bank of France,François Villeroy de Galhau: The role of centr...,Keynote address (virtual) by Mr François Ville...,https://www.bis.org/review/r200925c.htm
7,8,0,NaN,We are aware of the fact that for some countri...,11/17/2015,Sabine Lautenschläger,European Central Bank,Sabine Lautenschläger: Qualitative and quantit...,"Speech by Ms Sabine Lautenschläger, Member of ...",https://www.bis.org/review/r151124c.htm
8,9,0,NaN,The legislation that underpins insurance activ...,12/10/2009,Ewart S Williams,Central Bank of Trinidad and Tobago,Ewart S Williams: Preparing enhanced insurance...,"Speaking notes by Mr Ewart S Williams, Governo...",https://www.bis.org/review/r091221f.pdf
9,10,1,climate change,"As we gather in this room today, the world out...",8/2/2008,Boediono,Bank Indonesia,Boediono: Macroeconomic impact of climate chan...,"Keynote speech by Prof Dr Boediono, Governor o...",https://www.bis.org/review/r080901c.pdf


## 3. Define the prompt

In [ ]:
instructions = """
    You are coding excerpts from central bank speeches. Read the excerpt and decide how it frames climate change (including "green transition" or "green finance").

Choose ONE category:

N. NOT ABOUT CLIMATE: the excerpt does not discuss climate change.
A. PASSING MENTION: climate change appears only as one item in a list of challenges or topics, and nothing specific is said about it.
B. FINANCIAL RISK: climate change as a risk to banks, insurers or the financial system (supervision, stress tests, disclosure, risk management).
C. MACROECONOMIC: climate change as a driver of economic outcomes (inflation, growth, prices, harvests, uncertainty for policymakers).
D. INVESTMENT & OPPORTUNITY: financing the transition, green finance markets, innovation, business opportunities.
E. COLLECTIVE ACTION: climate change as a global or societal problem that needs cooperation, government policy or moral responsibility (market failure, multilateralism, future generations, fairness between countries).

Rules:
- Choose A whenever climate change is only listed alongside other issues, even if the list is about risks or the economy.
- If more than one of B–E fits, choose the one the excerpt emphasizes most.

Do NOT explain your answer. Respond ONLY with the letter that corresponds with the chosen category. For example: "A"

This is the excerpt:


    """

## 4. The message object

Like with the Anthropic playground, there are three roles: system, user, and assistant. We will load our prompt and text to be classified as the user message, and ask Anthropic to return to us the classification as the assistant. We do this because (1) it won't matter for our task whether our prompt is loaded as a system or user message, the model will get it either way, and (2) not all models have three functions, some only have two, so it is simpler to keep the coding consistent.

system: 'system' sits outside the message object and is a high-level direction around model behavior.

user: the user (you!) sending particular instructions and requests to the model

assistant: the model responding to the user

The message object records the conversation in a list, with each contribution attributed to a particular role (e.g., user or assistant).

A completed messages object (with the model's response) would look like this:

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Act as a translator from German to English. Here is the text: Wie geht es Ihnen?"
    },
    {
        "role": "assistant",
        "content": "How are you?"
    }
]

## 5. Try automating a single example


First, we'll save a single example in an object called "excerpt"

In [ ]:
excerpt = """
There are a wealth of initiatives at present: the G20, the United Nations, the Financial Stability Board and the OECD are all involved. And notable within the EU, undoubtedly, is the European Commission's action plan on sustainable finance. In the case of Spain, we should mention the Draft Climate Change Law.

"""

Next, let's fill in the messages object with what we will provide the model in the user role: the instructions and excerpt to be coded. The model will respond as the "assistant"; we shouldn't specify this role in advance.

In [ ]:
messages = [
    {
        "role": "user",
        "content": instructions + excerpt
    }
]

Now we need to create a client, which will send requests to the Anthropic API using our API key, and also receive responses and transmit them back to us. We create this using the api_key object we set-up at the start of this script.

In [ ]:
client = anthropic.Anthropic(
        api_key=api_key
    )

Now we will send a request to Claude to code the excerpt. We use the "client.messages.create" command to do this, and we save the result in an object called "response". "messages" here calls the object we just saved in the last step.

Note here that we need to specify which model we want to use. You can find a list of models and their official model API IDs here: https://platform.claude.com/docs/en/models/overview

In [ ]:
client = anthropic.Anthropic(
        api_key=api_key
    )

response = client.messages.create(
    model="claude-sonnet-5-5",
    messages=messages,
    max_tokens = 5000 # set max tokens to control costs and model reasoning
)

The resposne object, on its own, is a bit messy. Let's have a look. It contains some useful information, including the assistant's answer to our API request, but also token usage, which is helpful to note in terms of managing costs.

In [ ]:
print(response)

Message(id='msg_011CgRUTAgxv5sRwx1vJSZ5M', container=None, content=[TextBlock(citations=None, text='E', type='text')], diagnostics=None, model='claude-sonnet-5-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=578, output_tokens=3, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard'))


To get just the answer we're looking for, we can extract the relevant text. The response's `content` is a list of blocks, and the command below does two things: `[0]` selects the first block in that list (Python counts from zero), and `.text` pulls the text out of that block.

In [ ]:
response.content[0].text

'E'

## 6. Automate a bigger sample with a loop


The process here is exactly the same as in step 5. However, we put it in a loop instead, so that the client will run every one of the "text" column entries in our climate_dataset_10 file, and save the results to a csv file.

In [ ]:
# Initialize an empty list to store Claude outputs
Claude_outputs = []

# Iterate over each row in the dataframe
for index, row in climate_dataset_10.iterrows():
    # Extract the text from the dataframe
    text = row['text']

    messages = [
        {
            "role": "user",
            "content": instructions + text
        }
    ]

    # Classify, then store the result
    response = client.messages.create(
            model="claude-sonnet-5-5",
            messages=messages,
            max_tokens=5000
        )

        # Extract the response from Claude
    corrected_text = response.content[0].text

        # Append the classification to the list
    Claude_outputs.append(corrected_text)


# Print the number of rows classified
print(f"Number of rows classified: {len(Claude_outputs)}")

# Save outputs in a new column in our dataframe
climate_dataset_10['code_sonnet'] = Claude_outputs

Number of rows classified: 10


In [ ]:
climate_dataset_10

,excerpt_id,contains_keyword,keywords_found,text,date,speaker,central_bank,title,description,url,code_sonnet
0,1,0,NaN,Volatility is currently high in global capital...,11/4/2015,Benoît Cœuré,European Central Bank,Benoît Cœuré: Interview in El Financiero,"Interview with Mr Benoît Cœuré, Member of the ...",https://www.bis.org/review/r151117e.htm,N
1,2,1,climate change,There are a wealth of initiatives at present: ...,7/1/2019,Margarita Delgado,Bank of Spain,Margarita Delgado: Banks and society. Looking ...,"Speech by Ms Margarita Delgado, Deputy Governo...",https://www.bis.org/review/r190716h.htm,E
2,3,1,climate change,Only through robust and inclusive internationa...,10/1/2025,Jose Luis Escrivá,Bank of Spain,José Luis Escrivá: Introductory remarks - 9th ...,"Introductory remarks by Mr José Luis Escrivá, ...",https://www.bis.org/review/r251002e.htm,A
3,4,1,climate change,"But much remains to be done. In this respect, ...",7/16/2019,François Villeroy de Galhau,Bank of France,François Villeroy de Galhau: 75 years later - ...,Welcome address by Mr François Villeroy de Gal...,https://www.bis.org/review/r190716k.htm,A
4,5,0,NaN,The implementation of the eurosystem's asset p...,5/9/2018,Philip R Lane,Central Bank of Ireland,Philip R Lane: Publication of the Annual Repor...,"Introductory statement by Mr Philip R Lane, Go...",https://www.bis.org/review/r180727c.htm,N
5,6,1,climate change,"Rising protectionism, shifting geopolitical al...",9/16/2025,Yannis Stournaras,Bank of Greece,Yannis Stournaras: Greece as a strategic inves...,"Speech by Mr Yannis Stournaras, Governor of th...",https://www.bis.org/review/r250918d.htm,A
6,7,1,climate change,"Let me now come to the ECB's strategic review,...",9/25/2020,François Villeroy de Galhau,Bank of France,François Villeroy de Galhau: The role of centr...,Keynote address (virtual) by Mr François Ville...,https://www.bis.org/review/r200925c.htm,A
7,8,0,NaN,We are aware of the fact that for some countri...,11/17/2015,Sabine Lautenschläger,European Central Bank,Sabine Lautenschläger: Qualitative and quantit...,"Speech by Ms Sabine Lautenschläger, Member of ...",https://www.bis.org/review/r151124c.htm,N
8,9,0,NaN,The legislation that underpins insurance activ...,12/10/2009,Ewart S Williams,Central Bank of Trinidad and Tobago,Ewart S Williams: Preparing enhanced insurance...,"Speaking notes by Mr Ewart S Williams, Governo...",https://www.bis.org/review/r091221f.pdf,N
9,10,1,climate change,"As we gather in this room today, the world out...",8/2/2008,Boediono,Bank Indonesia,Boediono: Macroeconomic impact of climate chan...,"Keynote speech by Prof Dr Boediono, Governor o...",https://www.bis.org/review/r080901c.pdf,C


In [ ]:
# Export to csv and save locally
climate_dataset_10.to_csv(project_root/'sonnet_classification.csv', index=False)
files.download(str(project_root/'sonnet_classification.csv'))

## 7. Other options in client.messages.create()

There are several other ways you can customize your request to the Anthropic API. One of particular interest is effort.

From the documentation: "The effort parameter lets you control how many tokens Claude spends when responding to requests. You can trade off between response thoroughness and token efficiency". Details: https://platform.claude.com/docs/en/build-with-claude/effort

Possible effort settings: low, medium, high, xhigh, max (high is default on most models)

In [ ]:
response = client.messages.create(
    model="claude-sonnet-5-5",
    messages=messages,
    max_tokens = 5000, # set max tokens to control costs and model reasoning
    output_config={"effort": "max"}
)